In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, recall_score
)

NOTEBOOK_DIR = Path('.').resolve()
DATA_DIR     = NOTEBOOK_DIR

FEATURES    = ['temperature', 'humidity', 'tvoc_ppb', 'eco2_ppm']
CLASS_NAMES = ['Normal', 'Cooking', 'Fire']
RS          = 42


In [25]:
df = pd.read_excel(DATA_DIR / 'merged_dataset.xlsx')

print('Shape:', df.shape)
print('Class distribution:')
for name in CLASS_NAMES:
    n = (df['label'] == name).sum()
    print(f'  {name:8s}: {n:6d} ({n/len(df)*100:.1f}%)')


Shape: (60212, 6)
Class distribution:
  Normal  :  14671 (24.4%)
  Cooking :   4718 (7.8%)
  Fire    :  40823 (67.8%)


In [26]:
X = df[FEATURES].values
y = df['class_id'].values

# 70 / 15 / 15 stratified split
# test set is locked — not seen until final evaluation
X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RS, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.15/0.85, random_state=RS, stratify=y_tv)

print(f'Train : {len(y_train):,}')
print(f'Val   : {len(y_val):,}')
print(f'Test  : {len(y_test):,}')
print()
print('Train class distribution:')
for cid, name in enumerate(CLASS_NAMES):
    n = (y_train == cid).sum()
    if n: print(f'  {name}: {n:,} ({n/len(y_train)*100:.1f}%)')
print()
print('Test class distribution (stratified = proportional):')
for cid, name in enumerate(CLASS_NAMES):
    n = (y_test == cid).sum()
    if n: print(f'  {name}: {n:,} ({n/len(y_test)*100:.1f}%)')


Train : 42,148
Val   : 9,032
Test  : 9,032

Train class distribution:
  Normal: 10,270 (24.4%)
  Cooking: 3,302 (7.8%)
  Fire: 28,576 (67.8%)

Test class distribution (stratified = proportional):
  Normal: 2,201 (24.4%)
  Cooking: 708 (7.8%)
  Fire: 6,123 (67.8%)


In [ ]:
# scaler fitted on train only — transform val and test with same scaler
# fitting on all data would leak test distribution into training
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print('Scaler fitted on train only.')
print('Means:', dict(zip(FEATURES, scaler.mean_.round(2))))
print('Stds :', dict(zip(FEATURES, scaler.scale_.round(2))))


In [ ]:
# Baseline Gradient Boosting (replaces RF baseline)
rf_base = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=RS,
)
rf_base.fit(X_train_sc, y_train)
print('Baseline GB trained.')


In [ ]:
y_val_pred = rf_base.predict(X_val_sc)

print('=== Baseline — validation set ===')
print(classification_report(y_val, y_val_pred,
      target_names=CLASS_NAMES, labels=[0,1,2], zero_division=0))

cm_val = confusion_matrix(y_val, y_val_pred, labels=[0,1,2])
print(f'Fire recall (priority 1)           : {cm_val[2,2]/cm_val[2].sum():.4f}')
print(f'Cooking → Fire errors (priority 2) : {cm_val[1,2]}')


In [ ]:
# GridSearchCV on a 10k stratified subsample of train — avoids 45k+ row slowdown
# best params are then used to retrain on the full training set
rng = np.random.RandomState(RS)
idx = np.concatenate([
    rng.choice(np.where(y_train==cid)[0],
               min(len(np.where(y_train==cid)[0]), int(10000*len(np.where(y_train==cid)[0])/len(y_train))),
               replace=False)
    for cid in [0,1,2]
])
X_gs, y_gs = X_train_sc[idx], y_train[idx]
print(f'GridSearch subsample: {len(y_gs)} rows (class-balanced)')

param_grid = {
    'n_estimators'     : [100, 200],
    'learning_rate'    : [0.1, 0.05],
    'max_depth'        : [3, 5],
    'min_samples_split': [2, 5],
    'subsample'        : [0.8, 1.0],
}

# f1_macro: equal weight to all 3 classes — prevents ignoring the smaller Cooking class
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RS)
grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=RS),
    param_grid, cv=skf, scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_search.fit(X_gs, y_gs)

print('Best params    :', grid_search.best_params_)
print('Best CV F1-macro:', round(grid_search.best_score_, 4))


In [ ]:
# retrain best config on full training set
rf_best = GradientBoostingClassifier(
    **grid_search.best_params_,
    random_state=RS,
)
rf_best.fit(X_train_sc, y_train)

y_val_pred_tuned = rf_best.predict(X_val_sc)

print('=== Tuned GB — validation set ===')
print(classification_report(y_val, y_val_pred_tuned,
      target_names=CLASS_NAMES, labels=[0,1,2], zero_division=0))

cm_val_t = confusion_matrix(y_val, y_val_pred_tuned, labels=[0,1,2])
print(f'Fire recall (priority 1)           : {cm_val_t[2,2]/cm_val_t[2].sum():.4f}')
print(f'Cooking → Fire errors (priority 2) : {cm_val_t[1,2]}')


In [ ]:
# final evaluation — test set seen only once, here, at the end
y_test_pred = rf_best.predict(X_test_sc)

print('=== FINAL — test set (unseen data) ===')
print(classification_report(y_test, y_test_pred,
      target_names=CLASS_NAMES, labels=[0,1,2], zero_division=0))

cm_test = confusion_matrix(y_test, y_test_pred, labels=[0,1,2])
print(f'Fire recall (priority 1)           : {cm_test[2,2]/cm_test[2].sum():.4f}')
print(f'Cooking → Fire errors (priority 2) : {cm_test[1,2]}')
print(f'Macro F1                           : {f1_score(y_test, y_test_pred, average="macro", zero_division=0):.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (y_true, y_pred, title) in zip(axes, [
        (y_val,  y_val_pred_tuned, 'Validation set'),
        (y_test, y_test_pred,      'Test set (final)'),
]) :
    cm = confusion_matrix(y_true, y_pred, labels=[0,1,2])
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, colorbar=False)
    ax.set_title(title)
plt.tight_layout()
plt.show()


In [ ]:
# per-class recall shows how fairly the model performs across all 3 classes
# macro recall gives equal weight to each class regardless of size
recalls = recall_score(y_test, y_test_pred, labels=[0,1,2], average=None, zero_division=0)

plt.figure(figsize=(6, 4))
bars = plt.bar(CLASS_NAMES, recalls, color=['steelblue','tomato','gold'], alpha=0.85)
plt.ylim(0, 1.15)
plt.ylabel('Recall')
plt.title('Per-class recall — test set')
for bar, val in zip(bars, recalls):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.02,
             f'{val:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
fi = pd.Series(rf_best.feature_importances_, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(7, 4))
fi.plot.bar(color='steelblue', alpha=0.85)
plt.title('Feature importance')
plt.ylabel('Importance')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(fi.round(4).to_string())


In [ ]:
train_f1 = f1_score(y_train, rf_best.predict(X_train_sc), average='macro', zero_division=0)
val_f1   = f1_score(y_val,   y_val_pred_tuned,             average='macro', zero_division=0)
test_f1  = f1_score(y_test,  y_test_pred,                  average='macro', zero_division=0)
gap      = train_f1 - test_f1

print(f'Train F1-macro : {train_f1:.4f}')
print(f'Val   F1-macro : {val_f1:.4f}')
print(f'Test  F1-macro : {test_f1:.4f}')
print(f'Train-test gap : {gap:.4f}')
print('no overfitting' if gap < 0.02 else 'mild overfitting' if gap < 0.05 else 'overfitting detected')


In [ ]:
joblib.dump(rf_best, NOTEBOOK_DIR / 'model_gb.joblib')
joblib.dump(scaler,  NOTEBOOK_DIR / 'scaler.joblib')

print('Saved model_gb.joblib')
print('Saved scaler.joblib')
print('Best params:', grid_search.best_params_)
